# READ FIRST: this notebook was re-purposed for the Path B transfer test (Nov 11, 2026)

The sealed machinery below (environment, load, token resolution,
manual forward, scoring) is unchanged from the MiniCPM behavioral
notebook. What changed is only the battery flow: instead of the 360
synthesized tones, the runner asks 18 real-sound presence questions
and scores each from the model's own next-token numbers (the yes
share). Design frozen in `kill-test-wrapper/DESIGN-FROZEN-PATHB.md`.

**Files to upload before Step 3:** the 10 WAV files granted with
this run go in the session folder `/content/pathb_clips`:
swap_water.wav, swap_vacuum.wav, swap_beeps.wav, swap_sizzle.wav,
swap_speech.wav, swap_mixed.wav, swap_silence.wav,
series2_gray_water.wav, series2_gray_vacuum.wav,
series2_gray_beeps.wav. Step 2's cell creates that folder; upload
with the Colab Files panel into it.

**Output:** `pathb_minicpm_results.csv` in
`/content/project_k_outputs`, collected through the Files panel.
Older behavioral-era text further down (tones, six forms, 2,160
asks) is leftover wording from the original notebook and does NOT
describe this run. The runner's own prints and the CSV are the record.

# Project K, Series Three: asking MiniCPM-o 2.6 (behavioral questions)

**Why this notebook exists.** Series Three froze its
recipe on Nov 4, 2026 (PROTOCOL.md in
series-three-inside-tests), and MiniCPM-o 2.6 is now
running the same battery as the second model. Cell 1
(the codec control) and cell 2 (the probes, the tracing
notebook) are done for this model. This is cell 3,
**behavioral questions**: stop reading the model's
insides and ask the model the facts directly.

**The goal.** The 360 factorial property tones (pitch
500 vs 1500 Hz crossed with onset 0.5 vs 2.5 s, 90 per
cell, the identical synthesis and seed as Qwen's
property battery) are asked six questions each: is the
tone high-pitched, does it start in the first second,
the same two facts flipped (low-pitched, starts after
the first second), and the same two facts as forced
choices (high or low, early or late). 2,160 asks. Every
ask is scored three ways, the protocol's three: by the
answer word, by the margin at the fixed 0.5 cut, and by
AUC, the threshold-free separation. Margin claims name
their wording: there are six wordings here, and each
form's margin lives in its own column of the record.

**How an ask is scored.** MiniCPM's `model.chat`
exposes no yes score (the Series Two fact), so the
runner scores from the manual forward, the tracing
notebook's proven path: the chat preprocessing is
replicated step for step by hand, the embeddings run
through `get_vllm_embedding` and `get_omni_embedding`
into `model.llm`, and the next-token logits at the
final position are the readout. No text is generated.
The answer words (Yes, No, high, low, early, late)
are resolved from MiniCPM's own tokenizer before any
ask, as token sets, the exact construction Qwen's
frozen scorer used: each word's bare and
leading-space forms are encoded, the forms that are
exactly one token form the set, every candidate is
printed, and a word's score is the sum of the
full-softmax probabilities over its set. If a word
has no single-token form at all, its set is empty
and the run stops (exit code 11) before a single
margin exists.

**The feed subtlety, in the open.** Inside `chat`,
meeting a numpy audio array sets `use_tts_template=True`
no matter what the caller passed, so every audio feed
is templated with the model's default TTS chat
template. Series Two's nominal `use_tts_template=False`
never took effect on a feed that carried audio. This
runner replicates the behavior that actually ran, and
prints the first feed's full prompt text into the run
log as the record.

**Predictions, registered before the run.**

- P1 (word level): MiniCPM's spoken yes/no answers
  lean yes (its codec run answered Yes to a tone on
  pure silence, in both arms), so at the fixed 0.5 cut
  at least one of the two original questions scores
  near its base rate.
- P2 (margin level): if MiniCPM's reporting loss
  mirrors Qwen's threshold story, the yes/no margin
  separates pitch with AUC at or above 0.95 on the
  original wording even where the spoken word fails; a
  low AUC would place MiniCPM's loss earlier than
  Qwen's, and either outcome decides the comparison.
- P3 (wording): on the forced-choice forms the model
  picks the first-named option more often than chance,
  as Qwen did.

**Where it runs.** The paid Colab account (kobbytti),
where MiniCPM has always run, on an L4. The model loads
in bf16 only: precision never drops. **How to run it.**
Run the cells from the top, in order. Step 1 builds the
private environment and proves the stack. Step 2
prepares the folders (there is nothing to upload: the
tones are synthesized inside the runner from the fixed
seed). Step 3 writes the runner and runs its check mode
(proof, load, token resolution, sanity asks). Step 4
runs the battery. **Outputs** land in
`/content/project_k_outputs` only; this notebook never
mounts Drive, and the files are collected afterwards
through the Colab Files panel.

**The one rule.** We claim only what we ran: these
sounds, this model, these questions, this run.

## Step 1 · Setup: a private environment for the model's stack

The next cell does four things:

1. Installs `uv`, a fast environment and package tool, if this machine
   does not already have it. That is the only thing it adds to Colab's
   own Python, and it is a tool, not part of the model's stack.
2. Creates a private virtual environment at `/content/minicpm-venv`,
   on Python 3.12, the same Python generation Colab itself runs. The
   model's stack was proven to install cleanly on Python 3.12, so the
   private environment keeps 3.12 and changes only who owns it.
3. Installs the model's stack inside that environment, in two passes.
   First, torch and torchaudio pinned to 2.8.0 from PyTorch's CUDA 12.6
   shelf: that is the exact build this runtime already runs, so the
   chip driver match is the proven one, and the installer cannot drag
   in a multi-gigabyte CUDA 13 torch the driver may not support (left
   floating, the resolver does exactly that). Second, the model card's
   own pin list, unchanged: transformers 4.44.2, accelerate 0.34.2,
   bitsandbytes 0.46.1, librosa 0.10.2.post1 (a newer patch than the
   card lists, because the card's version predates Colab's current
   numpy), soundfile, vocos, and vector-quantize-pytorch. Pillow rides
   along because the model's code and the battery both need it and
   nothing in the pin list pulls it in. vocos and
   vector-quantize-pytorch are parts the model's code imports for its
   speech side; they are installed even though this test never asks
   the model to speak. One more install rides after those two passes:
   flash-attn 2.8.3, the attention library the model's own modeling
   file imports. Transformers checks that file's imports before it
   loads a single weight, so without flash-attn the model cannot
   even start. Colab's stock environment carries it already, which
   is why the model card never lists it; the private environment
   gets its own copy as the official prebuilt wheel for this exact
   stack (Python 3.12, torch 2.8, CUDA 12, cxx11 ABI), pinned by its
   full download address, never a build from source.
4. Proves the stack, in a fresh process under the environment's own
   Python. The proof prints the Python version, the torch version, the
   transformers version and the file it loads from, and it imports
   `WHISPER_ATTENTION_CLASSES` from the transformers whisper module,
   the exact symbol the model's own code asks for and newer
   transformers versions removed, and it imports flash_attn and
   prints its version. A version number alone is not proof anymore;
   the imports are. If any line of the proof is wrong, Step 1 stops
   on the spot and the run goes no further.

Nothing in this cell installs over, downgrades, or removes anything
Colab shipped. The two failed runs of this notebook both died in that
fight; this design simply refuses to have it.

The install downloads a few gigabytes (torch is the big one) and takes
several minutes. That is normal. Every command is printed before it
runs, and the installers stream their output into the cell, so nothing
about this step happens off the record.

In [ ]:
# --- Setup: build the private environment and prove the stack inside it --------
# This cell never installs the model's stack into Colab's Python. Two
# earlier runs of this notebook died doing exactly that: the pinned
# transformers never landed where the session reads. The stack now
# lives in a private virtual environment at /content/minicpm-venv, and
# Step 1 ends by proving, in a fresh process under that environment's
# Python, that the pinned versions and the one symbol the model's code
# needs are really there.
import os, sys, shutil, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VENV = "/content/minicpm-venv"
VENV_PY = os.path.join(VENV, "bin", "python")

def run_stream(cmd):
    print("+", " ".join(cmd))
    proc = subprocess.run(cmd)
    print("  exit code:", proc.returncode)
    return proc.returncode

# 1. uv: the environment tool. The only addition to Colab's own Python.
if shutil.which("uv") is None:
    print("uv not found. Installing it (a tool only, not the model stack).")
    rc = run_stream([sys.executable, "-m", "pip", "install", "uv"])
    if rc != 0 or shutil.which("uv") is None:
        print()
        print("!" * 72)
        print("STOP. uv could not be installed, so the private environment")
        print("cannot be built. Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: uv install failed")
print("uv:", shutil.which("uv"))

# 2. The private environment itself, on Python 3.12. The card's stack
# was proven to install cleanly on Python 3.12 (a full clean resolution
# of the exact pin list), so the private environment keeps 3.12 and
# changes only who owns the environment: this run, not Colab.
if not os.path.exists(VENV_PY):
    rc = run_stream(["uv", "venv", "--python", "3.12", VENV])
    if rc != 0:
        print()
        print("!" * 72)
        print("STOP. The private environment could not be created.")
        print("Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: venv creation failed")
ver = subprocess.run([VENV_PY, "--version"], capture_output=True, text=True)
print("venv python:", (ver.stdout or ver.stderr).strip())

# 3a. torch first, pinned to the runtime's own proven build. Colab's
# pip list on the failed runs showed torch 2.8.0+cu126: that is the
# build this chip and driver already run. Left floating, the resolver
# picks a newest torch built on CUDA 13 and downloads gigabytes of
# driver libraries the runtime may not support. Pinning it is not a
# change to the model card's list; the card leaves torch to the
# machine, and this makes the machine's proven choice explicit.
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "torch==2.8.0", "torchaudio==2.8.0",
                 "--index-url", "https://download.pytorch.org/whl/cu126"])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The torch install inside the private environment failed.")
    print("The installer's own output is above. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: torch install failed in venv")

# 3b. The model card's pin list, unchanged, in one resolver pass. No
# force-reinstall: the environment is empty and private, so there is
# nothing to force against. Pillow is added because the model's code
# and the battery import it and no package in the list pulls it in.
CARD_PACKAGES = [
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "bitsandbytes==0.46.1",
    "librosa==0.10.2.post1",
    "soundfile==0.13.1",
    "vocos==0.1.0",
    "vector-quantize-pytorch==1.18.5",
    "Pillow",
]
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY] + CARD_PACKAGES)
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The model card's stack did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: card stack install failed in venv")

# 3c. flash-attn: the one package the sealed environment was missing.
# MiniCPM's own modeling file imports flash_attn, and transformers
# checks that file's imports before it loads a single weight. The
# fifth round's full-capture log (minicpm_load_log.txt) shows all
# three rungs dying in about three seconds on exactly that check:
# "This modeling file requires the following packages that were not
# found in your environment: flash_attn". Colab's stock environment
# carries flash-attn 2.8.3 already, which is why the model card never
# lists it; the private environment has to be given its own copy. It
# goes in as the official prebuilt wheel, never a build from source:
# Dao-AILab's flash-attn 2.8.3 for CPython 3.12, torch 2.8, and CUDA
# 12, in the cxx11 ABI TRUE form, which is the ABI torch 2.8 itself
# ships with (PyTorch dropped its pre-cxx11 ABI builds at 2.8). The
# address was fetched and the wheel opened when this round was
# built: it answers HTTP 200, it holds the cp312 CUDA extension, and
# its version reads 2.8.3. Its one extra dependency, einops, is
# small and comes from the normal shelf in the same pass.
FLASH_ATTN_WHEEL = ("https://github.com/Dao-AILab/flash-attention/"
                    "releases/download/v2.8.3/flash_attn-2.8.3"
                    "%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312"
                    "-linux_x86_64.whl")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 FLASH_ATTN_WHEEL])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The flash-attn wheel did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: flash-attn install failed in venv")

# 4. Proof, in a fresh process under the environment's Python. Version
# alone is not accepted as proof anymore: the proof also imports
# WHISPER_ATTENTION_CLASSES, the exact symbol MiniCPM's own code asks
# transformers for, whose absence is what a wrong transformers version
# actually breaks, and flash_attn, the package the model's modeling
# file demands at the door (see 3c above).
PROOF = (
    "import sys, torch, transformers\n"
    "print('venv python:', sys.version.split()[0])\n"
    "print('torch version:', torch.__version__)\n"
    "print('transformers version:', transformers.__version__)\n"
    "print('transformers file:', transformers.__file__)\n"
    "from transformers.models.whisper.modeling_whisper import "
    "WHISPER_ATTENTION_CLASSES\n"
    "print('WHISPER_ATTENTION_CLASSES import: OK', "
    "sorted(WHISPER_ATTENTION_CLASSES.keys()))\n"
    "import flash_attn\n"
    "print('flash_attn version:', flash_attn.__version__)\n"
)
proof = subprocess.run([VENV_PY, "-c", PROOF],
                       capture_output=True, text=True)
print()
print(proof.stdout)
if proof.stderr.strip():
    print("proof process also said:")
    print("\n".join(proof.stderr.splitlines()[-10:]))
ok = ("transformers version: 4.44.2" in proof.stdout
      and "WHISPER_ATTENTION_CLASSES import: OK" in proof.stdout
      and "flash_attn version: 2.8.3" in proof.stdout)
if not ok:
    print("!" * 72)
    print("STOP. THE PROOF FAILED INSIDE THE PRIVATE ENVIRONMENT.")
    print("Either a version is wrong (transformers 4.44.2, flash_attn")
    print("2.8.3) or one of the proof imports failed: the whisper symbol")
    print("or flash_attn. Do not go on to Step 2. Show this to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: venv proof failed")
print("proof passed: transformers 4.44.2, loaded from the path above,")
print("WHISPER_ATTENTION_CLASSES imports from it, and flash_attn")
print("2.8.3 imports too.")

# The chip, read from the machine itself. This cell deliberately never
# imports torch or transformers into the notebook's own session.
smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print()
print((smi.stdout or smi.stderr).strip())
print("This notebook wants an L4 (about 24 GB). A T4 (about 16 GB)")
print("cannot hold the model in full form; the ladder would have to")
print("squeeze it, and the record would say so.")
print("Step 1 done. Go on to Step 2. No restart is needed.")

## Step 2 · Prepare the folders (nothing to upload)

Unlike the tracing battery, this run needs no master
files carried in. The 360 factorial tones and the 60
silence controls are synthesized inside the runner,
from the fixed seed 20261105, by code extracted
unchanged from the Qwen property builder: the same
draws, so the same waves Qwen was asked about.

The next cell only prepares the two folders: the clip
folder, and `/content/project_k_outputs`, the one home
for every output. This notebook never mounts Drive.
When the run is done, its files are collected from that
folder through the Colab Files panel.

In [ ]:
# --- Prepare the output and clip folders ---------------------------------------------
import os

for d in ("/content/project_k_outputs", "/content/behavioral_clips",
          "/content/pathb_clips"):
    os.makedirs(d, exist_ok=True)
    print("ready:", d)
print()
print("PATH B RUN: upload the 10 granted WAV files into")
print("/content/pathb_clips with the Files panel before Step 3,")
print("then go on to Step 3.")

## Step 3 · Write the runner, then check: proof, load, token resolution, sanity

The model never runs inside this notebook's cells
directly. It runs in a plain script,
`/content/minicpm_behavioral_run.py`, under the private
environment's Python. The first cell below writes that
script. The second runs it in **check mode**.

Check mode does five things, in order:

1. **Proof again.** The versions and files the runner
   imported, checked inside the private environment:
   transformers exactly 4.44.2 (or the run stops, exit
   code 3), and the whisper symbol MiniCPM's code needs.
2. **The load.** One rung: the pinned repo at the
   pinned revision, in bf16. There is no smaller rung
   behind it. If bf16 does not load, the run stops
   (exit code 4) with the full traceback in the run
   log.
3. **Token resolution.** The six answer words are
   resolved in MiniCPM's own tokenizer as token sets
   (Qwen's construction: the bare and leading-space
   forms that are exactly one token), every candidate
   printed. A word with no single-token form at all
   (an empty set) stops the run (exit code 11) before
   any ask.
4. **Clip generation.** The 420 clips are synthesized
   and measured (sample count, RMS, per-second RMS,
   waveform SHA-256 printed per clip; a failed
   measurement stops the run, exit code 9), and the
   manifest is saved.
5. **The sanity asks.** Two tones (one low-pitched, one
   high-pitched) and one silence clip go through all
   six question forms: 18 scored asks. Every feed
   prints its proof (decoded length and per-second
   RMS) before its forward; a broken feed stops the
   run (exit code 7). The sanity passes only if every
   logit is finite, every margin sits inside [0, 1],
   and every answer word falls inside its form's
   answer set; otherwise the run stops (exit code 12)
   and the battery must not run.

The battery does not run in check mode.

In [ ]:
# --- Write the MiniCPM behavioral runner script -----------------------------------
# The runner holds the proof, the load, the feed path (the
# tracing runner's manual forward), the six questions
# (byte-identical to Qwen's, substituted by the builder as
# reprs of the extracted strings), the token resolution,
# the clip generation (the Qwen property synthesis,
# unchanged), the sanity asks, and the battery. It is
# written to a plain file here and only ever run under the
# private environment's Python.
RUNNER_SOURCE = '\n# minicpm_behavioral_run.py - the Series Three behavioral\n# questions runner (PROTOCOL cell 3) for MiniCPM-o 2.6,\n# under the frozen protocol. Written by\n# project_k_minicpm_behavioral_nodrive_colab.ipynb. It\n# runs only under the private environment\'s Python\n# (/content/minicpm-venv/bin/python). Modes: "check" runs\n# the proof, the load, the token resolution, clip\n# generation, and the sanity asks (2 tones and 1 silence\n# clip through all 6 question forms, 18 scored asks),\n# then stops. "battery" does all of that and then asks\n# all 360 factorial tones all 6 forms: 2,160 scored\n# asks, scored by word, by margin at the fixed 0.5 cut,\n# and by AUC.\n#\n# The readout: MiniCPM\'s model.chat exposes no yes\n# score (the Series Two fact), so every ask is scored\n# from the manual forward, the tracing notebook\'s own\n# path: build the templated prompt embeds exactly as\n# chat does, run get_vllm_embedding and\n# get_omni_embedding into model.llm, and read the\n# next-token logits at the final position. No text is\n# generated, so there are no decoding settings; the\n# capture policy (one forward per ask, final-position\n# next-token logits only) is printed at the top of this\n# log, and the first feed\'s full templated prompt is\n# printed verbatim in the log as the prompt record.\n#\n# Exit codes, one per failure family, all loud:\n#   3  the stack proof failed (transformers is not\n#      exactly 4.44.2, the pin MiniCPM\'s stack needs)\n#   4  the model did not load (bf16 is the only rung;\n#      precision never drops)\n#   7  a feed returned no audio or a broken waveform\n#      (never padded, never scored)\n#   9  probe clip generation failed its own\n#      measurements (raised by the synthesis section)\n#   11 an answer word has no single-token surface\n#      form in MiniCPM\'s tokenizer, so its scoring\n#      token set is empty: the scoring tokens are\n#      not settled, so no ask is believed\n#   12 the sanity asks failed: a logit was not finite,\n#      a margin fell outside [0, 1], or the sanity ask\n#      count was wrong\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\n\n# --- Homes for the outputs -------------------------------------------------------\n# One home, on purpose: /content/project_k_outputs, the\n# no-Drive pattern from the Nov 7, 2026 notebooks. There\n# is no Drive mount anywhere in this notebook; the\n# outputs are collected later through the Colab Files\n# panel.\nOUT_HOME = "/content/project_k_outputs"\nos.makedirs(OUT_HOME, exist_ok=True)\n\ndef out_paths(name):\n    return [os.path.join(OUT_HOME, name)]\n\ndef save_both(name, write_fn):\n    saved = []\n    for p in out_paths(name):\n        try:\n            write_fn(p)\n            saved.append(p)\n            print("saved:", p)\n        except Exception as e:\n            print("could not save", p, "-", str(e)[:160])\n    return saved\n\n# --- The run log: every print also lands in the outputs folder -------------------\n# The Qwen index-check runner\'s tee, retargeted at the\n# outputs folder, so a stopped run\'s evidence never\n# depends on the notebook output area\'s scrolling. The\n# load\'s full tracebacks land here too, through the tee.\nclass _Tee:\n    def __init__(self, *streams):\n        self.streams = streams\n    def write(self, s):\n        for st in self.streams:\n            try:\n                st.write(s)\n            except Exception:\n                pass\n        return len(s)\n    def flush(self):\n        for st in self.streams:\n            try:\n                st.flush()\n            except Exception:\n                pass\ntry:\n    _log_fh = open(os.path.join(\n        OUT_HOME, "pathb_minicpm_run_log.txt"),\n        "w", buffering=1)\nexcept Exception:\n    _log_fh = open("/content/pathb_minicpm_run_log.txt",\n                   "w", buffering=1)\nsys.stdout = _Tee(sys.stdout, _log_fh)\n\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\nprint("entry point: minicpm_behavioral_run.py, run under "\n      "/content/minicpm-venv/bin/python")\nprint("capture policy: one manual forward per ask; the "\n      "next-token logits at the final position are the "\n      "only readout; no text is generated, so there are "\n      "no decoding settings to log")\nprint("prompt record: the first feed of the run prints "\n      "its full templated prompt text in this log; every "\n      "later feed names its question in the results CSV")\n\n# --- Proof lines, printed before anything loads ----------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nfrom transformers.models.whisper.modeling_whisper import WHISPER_ATTENTION_CLASSES\nprint("WHISPER_ATTENTION_CLASSES import: OK",\n      sorted(WHISPER_ATTENTION_CLASSES.keys()))\nif transformers.__version__ != "4.44.2":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.44.2,")\n    print("the version MiniCPM-o 2.6\'s own stack pins.")\n    sys.exit(3)\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()\n      else "none")\n\n# --- Load the model: one rung, bf16 ------------------------------------------------\n# The same repo, the same pinned revision, and the same\n# load call as the Series Two MiniCPM runner and the\n# tracing runner, with one deliberate property: there is\n# no 8-bit or 4-bit rung behind it. PROTOCOL: precision\n# never drops to fit a smaller GPU. If bf16 does not\n# load, the run stops here (exit 4) with the full\n# traceback printed into this run log through the tee.\nimport shutil, traceback\nfrom transformers import AutoModel, AutoProcessor\n\nREPO = "openbmb/MiniCPM-o-2_6"\nREVISION = "06849bfd36da94da1f5a88fa17e8ba63f08a4c4d"\n\ntry:\n    from transformers.dynamic_module_utils import HF_MODULES_CACHE\n    module_cache = os.path.join(str(HF_MODULES_CACHE), *REPO.split("/"))\n    if os.path.isdir(module_cache):\n        shutil.rmtree(module_cache)\n        clear_note = "cleared cached remote code for this repo: " + module_cache\n    else:\n        clear_note = ("no cached remote code for this repo to clear: "\n                      + module_cache)\nexcept Exception as e:\n    clear_note = "WARNING: the remote-code cache clear did not run: " + str(e)\nprint(clear_note)\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\nprocessor = None\nMODEL_NAME = None\nLOAD_MODE = None\n\nprint("=" * 70)\nprint(f"loading: {REPO} at pinned revision {REVISION}, bf16 only")\ntry:\n    m = AutoModel.from_pretrained(\n        REPO, revision=REVISION, torch_dtype=torch.bfloat16,\n        trust_remote_code=True, attn_implementation="sdpa",\n        init_vision=True, init_audio=True, init_tts=False)\n    m = m.eval().cuda()\n    p = AutoProcessor.from_pretrained(\n        REPO, revision=REVISION, trust_remote_code=True)\n    model, processor = m, p\n    MODEL_NAME, LOAD_MODE = REPO, "bf16"\n    print("loaded:", REPO, "in bf16 form")\nexcept Exception:\n    tb = traceback.format_exc()\n    print("the bf16 load failed. Full traceback:")\n    print(tb)\n    print()\n    print("!" * 70)\n    print("THE MODEL DID NOT LOAD IN BF16. There is no second")\n    print("rung in this notebook: precision never drops.")\n    print("The full traceback is in this run log, above.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\nLLM = model.llm\nprint("language model:", type(LLM).__name__)\nprint("llm config num_hidden_layers:",\n      getattr(LLM.config, "num_hidden_layers", "not exposed"))\nprint("llm config hidden_size:",\n      getattr(LLM.config, "hidden_size", "not exposed"))\nprint("model config audio_chunk_length:", model.config.audio_chunk_length)\nprint("model config chunk_input:", model.config.chunk_input)\n\n# --- The feed path -----------------------------------------------------------------\n# Sound alone, replicating model.chat\'s preprocessing\n# exactly, read from the pinned remote code\n# (modeling_minicpmo.py at the pinned revision), and\n# carried over from the tracing runner unchanged. chat\n# does four things before it generates, and this runner\n# does the same four by hand so the logits can be read:\n#\n#   1. It rewrites the messages: each numpy audio array\n#      in a message\'s content becomes the literal text\n#      "(<audio>./</audio>)", its message index is\n#      recorded in audio_parts, string content passes\n#      through, and the pieces are joined with "\\n"\n#      (omni_input=False). One forced side effect lives\n#      here: meeting an audio array sets\n#      use_tts_template=True inside chat, whatever the\n#      caller passed, so the prompt text below is built\n#      with the model\'s default TTS chat template. That\n#      is what actually ran in Series Two, whose call\n#      nominally passed use_tts_template=False; this\n#      runner replicates the behavior, and prints the\n#      first feed\'s full prompt text as the record.\n#   2. It builds the prompt text with the tokenizer\'s\n#      apply_chat_template (add_generation_prompt=True).\n#   3. It calls the processor with the prompt list, the\n#      image list (empty here), the audio list (the\n#      per-second chunk arrays), and audio_parts, with\n#      chunk_input=True, max_slice_nums=1,\n#      use_image_id=False, return_tensors="pt",\n#      max_length=32768, and moves the result to the\n#      model\'s device. It then pops "image_sizes", as\n#      chat does before generate.\n#   4. The model\'s own forward then runs\n#      get_vllm_embedding, get_omni_embedding with\n#      chunk_length=model.config.audio_chunk_length,\n#      and model.llm over the spliced embeddings. This\n#      runner calls those same three steps and reads\n#      the next-token logits. The processor\'s batch\n#      carries no position_ids (its converter never\n#      builds them); the model\'s generate builds them\n#      as a plain arange over the row, so this runner\n#      does the same, and asserts the single row is\n#      unpadded (attention_mask all true), which is\n#      what makes the arange exact.\n#\n# The message shape is the Series Two audio-alone shape:\n# the system prompt from get_sys_prompt(mode="default",\n# language="en"), then one user message whose content is\n# the clip as per-second chunk arrays followed by the\n# question. The questions in this battery are the\n# property questions defined below, byte-identical to\n# Qwen\'s; the presence questions in PROMPTS are the\n# tracing set, carried because the synthesis section\n# stamps them into the manifest, byte-identical to the\n# Qwen property runner\'s own PROMPTS.\n#\n# Two tracing readouts are deliberately absent here:\n# the audio positions and the tower embeddings. Cell 3\n# reads only the final position\'s next-token logits;\n# positions and the tower belong to the probe cell,\n# where the tracing notebook proved them on this same\n# stack.\nimport json, math\nimport numpy as np\nimport soundfile as sf\nimport librosa\n\nCLIPS = "/content/clips"\nSR = 16000\nCLIP_SAMPLES = SR * 4\nPROMPTS = {\n    "tone": ("Is there the sound of a tone in this clip? "\n             "Answer yes or no."),\n    "speech": ("Is there the sound of speech in this clip? "\n               "Answer yes or no."),\n    "silence": ("Is there the sound of a tone in this clip? "\n                "Answer yes or no."),\n}\n\nclass FeedError(RuntimeError):\n    pass\n\ndef rms_db_np(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x.astype(np.float64)))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef read_wave(path, expect_silence=False):\n    # Read one battery clip and prove the decoded waveform\n    # before anything is built from it: exact sample rate,\n    # exact length, RMS logged by the caller. A track that\n    # decodes to all zeros on a clip that should carry\n    # sound stops the run (the caller turns FeedError into\n    # exit 7): a broken feed is never padded and never\n    # scored.\n    if not os.path.exists(path):\n        raise FeedError(f"feed file missing: {path}")\n    wave, sr = sf.read(path, dtype="float32", always_2d=False)\n    if sr != SR:\n        raise FeedError(\n            f"{path} decoded at {sr} Hz, not {SR}; the battery "\n            f"clips are 16 kHz mono by construction")\n    wave = np.asarray(wave, dtype=np.float32)\n    if wave.ndim > 1:\n        wave = wave[:, 0]\n    if len(wave) != CLIP_SAMPLES:\n        raise FeedError(\n            f"{path} decoded to {len(wave)} samples, not "\n            f"{CLIP_SAMPLES}; the battery clips are exactly "\n            f"4.0 s by construction")\n    if not expect_silence and not np.any(wave != 0):\n        raise FeedError(\n            f"decoded track is all zeros for {path}, on a clip "\n            f"that should carry sound")\n    return wave\n\ndef build_inputs(wave, prompt, print_prompt=False):\n    # Steps 1 to 3 of chat\'s preprocessing, for one clip.\n    # Returns the processor\'s data dict, on the model\'s\n    # device, with image_sizes popped, as chat leaves it.\n    sys_msg = model.get_sys_prompt(mode="default", language="en")\n    unit_count = int(math.ceil(len(wave) / SR))\n    chunks = []\n    for i in range(unit_count):\n        chunk = wave[SR * i: SR * (i + 1)]\n        if len(chunk) < SR:\n            chunk = np.pad(chunk, (0, SR - len(chunk))).astype(np.float32)\n        chunks.append(chunk)\n    contents = list(chunks) + [prompt]\n    msgs = [sys_msg, {"role": "user", "content": contents}]\n    # chat\'s own rewrite of the messages, step for step.\n    prompts = []\n    audios = []\n    audio_parts = []\n    use_tts_template = False\n    for i, msg in enumerate(msgs):\n        content = msg["content"]\n        if isinstance(content, str):\n            content = [content]\n        cur = []\n        for c in content:\n            if isinstance(c, np.ndarray):\n                audios.append(c)\n                audio_parts.append(i)\n                cur.append("(<audio>./</audio>)")\n                # chat sets this itself when it meets audio;\n                # replicated, and named in the open above.\n                use_tts_template = True\n            elif isinstance(c, str):\n                cur.append(c)\n            else:\n                raise FeedError(\n                    f"unsupported content item in message {i}: "\n                    f"{type(c).__name__}")\n        prompts.append("\\n".join(cur))\n    template = (model.default_tts_chat_template\n                if use_tts_template else None)\n    text = processor.tokenizer.apply_chat_template(\n        [{"role": m["role"], "content": p}\n         for m, p in zip(msgs, prompts)],\n        tokenize=False, add_generation_prompt=True,\n        chat_template=template)\n    if print_prompt:\n        print("PROMPT TEXT for this feed (exact, as templated):")\n        print(text)\n    data = processor([text], [[]], [audios], [audio_parts],\n                     max_slice_nums=1, use_image_id=False,\n                     chunk_input=True, return_tensors="pt",\n                     max_length=32768).to(model.device)\n    data.pop("image_sizes", None)\n    return data\n\ndef llm_forward(data):\n    # Step 4: the model\'s own forward path, called by\n    # hand so the logits can be read. Returns the llm\n    # output (with logits). Runs under no_grad, as the\n    # tracing runner\'s forward did, so no returned tensor\n    # requires grad.\n    embeds, _ = model.get_vllm_embedding(data)\n    embeds = model.get_omni_embedding(\n        data, embeds,\n        chunk_length=model.config.audio_chunk_length)\n    mask = data["attention_mask"][0]\n    if not bool(mask.all()):\n        raise FeedError(\n            "the processor padded this row; the arange "\n            "position_ids this runner builds are only exact "\n            "for an unpadded row, and the battery feeds one "\n            "clip at a time")\n    seq_len = int(data["input_ids"].shape[1])\n    position_ids = torch.arange(\n        seq_len, dtype=torch.long, device=embeds.device).unsqueeze(0)\n    with torch.no_grad():\n        out = LLM(input_ids=None, position_ids=position_ids,\n                  inputs_embeds=embeds, output_hidden_states=True)\n    return out\n\n# --- The questions -------------------------------------------------------------------\n# Byte-identical to Qwen\'s, by construction: the builder\n# extracted each string from the Qwen builders\' own\n# source and substituted it here as a repr, so nothing\n# was retyped from memory. The originals are Part B of\n# build_indexcheck_qwen_notebook.py (lines 243 and 245);\n# the four wording forms are Part 2 of\n# build_closers_qwen_notebook.py (lines 170 to 177).\nPITCH_Q = \'Is the tone in this clip high-pitched? Answer yes or no.\'\nONSET_Q = \'Does the tone start in the first second of this clip? Answer yes or no.\'\nPITCH_REV_Q = \'Is the tone in this clip low-pitched? Answer yes or no.\'\nONSET_REV_Q = \'Does the tone start after the first second of this clip? Answer yes or no.\'\nPITCH_CHOICE_Q = \'Is the tone high-pitched or low-pitched? Answer with one word: high or low.\'\nONSET_CHOICE_Q = \'Does the tone start early or late in this clip? Answer with one word: early or late.\'\n\n# The six forms. A yes/no form\'s answer pair is Yes/No\n# and its margin is the yes share: p_yes over\n# (p_yes + p_no), where each word\'s p is the sum of\n# the full-softmax probabilities over that word\'s\n# token set, Qwen\'s frozen scorer\'s own construction.\n# A choice form\'s answer pair is its two named\n# options and its margin is the first-named option\'s\n# share of the same two summed probabilities.\n# Each form\'s truth maps a clip\'s pitch and onset bands\n# to the true answer word; margins in any claim must\n# name their wording (PROTOCOL cell 3).\nFORMS = [\n    {"name": "pitch", "question": PITCH_Q, "kind": "yesno",\n     "opt_a": "Yes", "opt_b": "No",\n     "truth": lambda pb, ob: "Yes" if pb == "high" else "No"},\n    {"name": "onset", "question": ONSET_Q, "kind": "yesno",\n     "opt_a": "Yes", "opt_b": "No",\n     "truth": lambda pb, ob: "Yes" if ob == "early" else "No"},\n    {"name": "pitch_reversed", "question": PITCH_REV_Q,\n     "kind": "yesno", "opt_a": "Yes", "opt_b": "No",\n     "truth": lambda pb, ob: "Yes" if pb == "low" else "No"},\n    {"name": "onset_reversed", "question": ONSET_REV_Q,\n     "kind": "yesno", "opt_a": "Yes", "opt_b": "No",\n     "truth": lambda pb, ob: "Yes" if ob == "late" else "No"},\n    {"name": "pitch_choice", "question": PITCH_CHOICE_Q,\n     "kind": "choice", "opt_a": "high", "opt_b": "low",\n     "truth": lambda pb, ob: "high" if pb == "high" else "low"},\n    {"name": "onset_choice", "question": ONSET_CHOICE_Q,\n     "kind": "choice", "opt_a": "early", "opt_b": "late",\n     "truth": lambda pb, ob: "early" if ob == "early" else "late"},\n]\n\n# --- Token resolution ------------------------------------------------------------------\n# The answer words are resolved in MiniCPM\'s own\n# tokenizer, at runtime, before any ask, as token\n# SETS: the exact construction Qwen\'s frozen scorer\n# used (YES_TOKEN_IDS and NO_TOKEN_IDS in the handover\n# rerun\'s head, summed over the full softmax in the\n# index check\'s forward_score). For every word, the\n# candidate surface forms (the bare word and its\n# leading-space form) are printed with the token ids\n# they encode to; the forms that are exactly one token\n# form the word\'s set, and the word\'s probability in\n# an ask is the sum of the full-softmax probabilities\n# over its set. The only hard stop here (exit 11) is a\n# word whose set is empty: a word with no single-token\n# form at all has no defined probability read.\n# (Correction, Nov 10, 2026: this section first\n# required exactly one token in exactly one form and\n# stopped when two single tokens claimed one word;\n# the check run stopped on "Yes" that way, bare token\n# 9454 and leading-space token 7414, both single\n# tokens. The set construction replaced it the same\n# day, so the token construction is now Qwen\'s,\n# exactly, not a parallel one.)\nANSWER_WORDS = ["Yes", "No", "high", "low", "early", "late"]\nTOK = {}\n\ndef token_fail(reason):\n    print()\n    print("!" * 70)\n    print("TOKEN RESOLUTION FAILED: " + reason)\n    print("The scoring tokens are not settled, so no ask "\n          "has run and no")\n    print("later number in this run can be trusted. "\n          "Show this whole")\n    print("output to Muse.")\n    print("!" * 70)\n    sys.exit(11)\n\ndef resolve_tokens():\n    tok = processor.tokenizer\n    print("=" * 70)\n    print("TOKEN RESOLUTION: the answer words in "\n          "MiniCPM\'s own tokenizer")\n    for word in ANSWER_WORDS:\n        ids = set()\n        for form in (word, " " + word):\n            enc = tok.encode(form, add_special_tokens=False)\n            print(f"  candidate {form!r}: token ids {enc}")\n            if len(enc) == 1:\n                ids.add(int(enc[0]))\n        if not ids:\n            token_fail(\n                f"{word!r} has no single-token surface "\n                f"form: neither its bare nor its "\n                f"leading-space form is exactly one "\n                f"token, so its scoring token set is "\n                f"empty and a probability read on it is "\n                f"not defined")\n        TOK[word] = sorted(ids)\n        print(f"  -> {word!r}: token id set {TOK[word]} "\n              f"(decodes to "\n              f"{[tok.decode([t]) for t in TOK[word]]})")\n    print("TOKEN RESOLUTION PASSED:", TOK)\n\nresolve_tokens()\n\n# --- Probe clip generation -------------------------------------------------------------\n# Round 2 fix 2. Fixed seed, so the run can be repeated\n# exactly and the split can be re-derived. 420 clips,\n# 360 per class (tone, speech, silence), each exactly\n# 4.0 s at 16 kHz mono (64000 samples), fed later as raw\n# WAV: no MP4, no codec anywhere.\n#\n# Variation inside each class, so a helper must read the\n# sound and not memorize one waveform:\n#   tone: a 0.5 s sine burst, frequency jittered 950 to\n#     1050 Hz, start 0.5 to 3.0 s, level -24 to -14 dBFS.\n#   speech: the probe sentence from the master, tape-style\n#     speed factor 0.92 to 1.08 (the resample shifts pitch\n#     with speed, the small speed/pitch variation), gain\n#     -6 to +3 dB, start offset up to 1.5 s.\n#   silence: nothing in the foreground at all.\n# Every clip, all classes alike, sits on a background drawn\n# from one shared scheme: plain digital silence, with a\n# one-in-six chance of a near-silence white-noise floor at\n# -75 to -65 dBFS. The silence class is therefore true\n# digital silence plus a few near-silence noise-floor\n# variants, and the background can never be a class cue,\n# because every class draws from the same scheme.\n#\n# The split is fixed by seed: 240 train / 120 held-out test\n# per class, stratified, the same split for every layer and\n# both tasks. split_ranks below is the single source of\n# truth: generation stamps each clip\'s split in the\n# manifest, and the probe step re-derives the same ranks and\n# asserts the two agree.\n#\n# Every clip is measured before anything uses it (sample\n# count, whole-clip RMS, per-second RMS, waveform SHA-256)\n# and printed: the feed proof. A failed measurement stops\n# the run (exit 9).\nimport csv, hashlib\n\nSEED = 20261105\nN_TRAIN = 240\nSR = 16000\nCLIP_SAMPLES = SR * 4\nCLASS_NUMBER = {"tone": 1, "speech": 2, "silence": 3}\nCLIP_DIRS = ["/content/behavioral_clips",\n             "/content/behavioral_clips"]\nfor d in CLIP_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\nMANIFEST_FIELDS = [\'clip_name\', \'group\', \'split\', \'file\', \'freq_hz\', \'start_s\', \'burst_s\', \'level_dbfs\', \'gain_db\', \'speed_factor\', \'floor_dbfs\', \'rms_dbfs\', \'wave_sha256\', \'prompt\', \'pitch_band\', \'onset_band\']\n\ndef split_ranks(class_name):\n    # The within-class ranks (0-based, in generation order)\n    # that belong to train and to test for one class.\n    rng = np.random.default_rng(SEED + 1009 * CLASS_NUMBER[class_name])\n    order = rng.permutation(360)\n    return (set(int(x) for x in order[:N_TRAIN]),\n            set(int(x) for x in order[N_TRAIN:]))\n\ndef rms_db_np(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x.astype(np.float64)))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef db_to_amp(db):\n    return float(10.0 ** (db / 20.0))\n\ndef wave_hash(wave):\n    return hashlib.sha256(\n        np.asarray(wave, dtype=np.float32).tobytes()).hexdigest()\n\ndef background(rng):\n    # One draw from the shared background scheme: (wave,\n    # floor level in dBFS or None for digital silence).\n    wave = np.zeros(CLIP_SAMPLES, dtype=np.float32)\n    if rng.random() < 1.0 / 6.0:\n        level = float(rng.uniform(-75.0, -65.0))\n        noise = rng.standard_normal(CLIP_SAMPLES).astype(np.float32)\n        cur = float(np.sqrt(np.mean(np.square(\n            noise.astype(np.float64)))))\n        wave = (noise * (db_to_amp(level) / cur)).astype(np.float32)\n        return wave, round(level, 2)\n    return wave, None\n\ndef write_clip(name, wave):\n    wave = np.asarray(wave, dtype=np.float32)\n    assert len(wave) == CLIP_SAMPLES, \\\n        f"clip {name} has {len(wave)} samples, not {CLIP_SAMPLES}"\n    local = os.path.join("/content/behavioral_clips", name)\n    drive = os.path.join(\n        "/content/behavioral_clips", name)\n    # ALWAYS write the wave in hand, to both homes, overwriting\n    # whatever is there. Never copy an existing file into place:\n    # /content is fresh each session but the Drive folder\n    # persists across runs, and an earlier run\'s file under the\n    # same name can hold a different wave (the draw sequence\n    # moved when the builder changed). That stale-file reuse is\n    # exactly what stopped the Nov 3 diagnostic run at clip\n    # 413: prop_silence_53\'s row described a -67.5 dB noise\n    # floor while the file on disk was an older run\'s digital\n    # silence. The read-back check in finish_row now proves\n    # the file on disk is the wave in hand before extraction\n    # ever sees it.\n    sf.write(local, wave, SR, subtype="PCM_16")\n    try:\n        sf.write(drive, wave, SR, subtype="PCM_16")\n    except Exception:\n        pass\n    return local\n\ndef speech_segment():\n    # The sentence inside the speech master: the region whose\n    # short-window RMS rises above -45 dBFS, with a small pad.\n    # Same cut as round 1.\n    path = os.path.join(CLIPS, "codec_speech_raw.wav")\n    wave, _ = librosa.load(path, sr=SR, mono=True)\n    wave = np.asarray(wave, dtype=np.float32)\n    win = SR // 100  # 10 ms\n    n = len(wave) // win\n    rms = np.array([rms_db_np(wave[i * win:(i + 1) * win])\n                    for i in range(n)])\n    above = np.where(rms > -45.0)[0]\n    if len(above) == 0:\n        raise RuntimeError(\n            "no active region found in the speech master; the "\n            "speech positives cannot be built")\n    start = max(0, int(above[0] * win) - SR // 20)\n    end = min(len(wave), int((above[-1] + 1) * win) + SR // 20)\n    seg = wave[start:end]\n    print(f"speech segment: {len(seg) / SR:.2f} s cut from the "\n          f"master at {start / SR:.2f} s to {end / SR:.2f} s")\n    if len(seg) > CLIP_SAMPLES:\n        seg = seg[:CLIP_SAMPLES]\n        print("speech segment trimmed to the 4.0 s window")\n    return seg\n\ndef tape_resample(x, speed):\n    # Tape-style speed change: playing the segment faster or\n    # slower, which shifts its pitch by the same factor. The\n    # round 2 speech variation (speed 0.92 to 1.08, about a\n    # semitone and a half either way at the extremes).\n    n_out = max(1, int(round(len(x) / speed)))\n    idx = np.linspace(0, len(x) - 1, n_out)\n    return np.interp(idx, np.arange(len(x)), x).astype(np.float32)\n\ndef finish_row(rows, name, group, rank, wave, prompt, extra):\n    # Peak guard, split stamp, measurement, and the printed\n    # feed proof for one generated clip.\n    peak = float(np.max(np.abs(wave))) if len(wave) else 0.0\n    if peak > 0.99:\n        wave = (wave * (0.99 / peak)).astype(np.float32)\n        print(f"  {name}: peak {peak:.2f} over the guard; scaled "\n              f"to a 0.99 peak")\n    path = write_clip(name, wave)\n    # Read-back proof: the file on disk must be the wave in\n    # hand (PCM_16 quantisation moves the RMS by far less\n    # than 1 dB at these levels; digital silence reads back\n    # at exactly -120). A mismatch stops generation here,\n    # where the cause is visible, instead of surfacing as a\n    # feed hard stop deep inside extraction.\n    _back, _ = sf.read(path, dtype="float32")\n    if len(_back) != len(wave) or abs(rms_db_np(_back)\n                                      - rms_db_np(wave)) > 1.0:\n        print()\n        print("!" * 70)\n        print(f"CLIP WRITE CHECK FAILED: {name} on disk does "\n              f"not match the generated wave (disk rms "\n              f"{rms_db_np(_back):.2f} dB vs wave rms "\n              f"{rms_db_np(wave):.2f} dB). Nothing was "\n              f"extracted. Show this whole output to Muse.")\n        print("!" * 70)\n        sys.exit(9)\n    train_ranks, _ = split_ranks(group)\n    split = "train" if rank in train_ranks else "test"\n    row = {"clip_name": name[:-4], "group": group, "split": split,\n           "file": path, "wave_sha256": wave_hash(wave),\n           "rms_dbfs": round(rms_db_np(wave), 2), "prompt": prompt,\n           "_expect_silence": bool(group == "silence"\n                                   and extra.get("floor_dbfs") in ("", None))}\n    row.update(extra)\n    split = row["split"]\n    sec = [rms_db_np(wave[k * SR:(k + 1) * SR]) for k in range(4)]\n    print(f"  {row[\'clip_name\']:<14} group={group:<7} split={split:<5} "\n          f"samples={len(wave)} rms={row[\'rms_dbfs\']:.2f} dB "\n          f"per-second [{\', \'.join(f\'{v:.1f}\' for v in sec)}] dB "\n          f"sha256={row[\'wave_sha256\'][:16]}")\n    if len(wave) != CLIP_SAMPLES:\n        print()\n        print("!" * 70)\n        print(f"CLIP MEASUREMENT FAILED: {row[\'clip_name\']} has "\n              f"{len(wave)} samples, not {CLIP_SAMPLES}. The "\n              f"generated set is not what the plan says. Nothing "\n              f"was extracted. Show this whole output to Muse.")\n        print("!" * 70)\n        sys.exit(9)\n    rows.append(row)\n\ndef generate_clips():\n    # The Handover Correction step 5 stimulus set. Factorial\n    # tones: pitch (500 vs 1500 Hz) crossed with onset (0.5 vs\n    # 2.5 s), 90 instances per cell, 360 tones; 60 silence\n    # clips (the shared background scheme alone). Every\n    # instance is a fresh draw (pitch jitter x0.98 to 1.02,\n    # onset jitter +-0.1 s, level -20 +-1 dBFS, random phase),\n    # so every waveform is a unique identity; the split is by\n    # instance within each cell (60 train / 30 test), stamped\n    # here and asserted unique by hash below.\n    rng = np.random.default_rng(SEED)\n    rows = []\n    cells = [("low", 500.0, "early", 0.5), ("low", 500.0, "late", 2.5),\n             ("high", 1500.0, "early", 0.5), ("high", 1500.0, "late", 2.5)]\n    for ci, (pband, pbase, oband, obase) in enumerate(cells):\n        order = rng.permutation(90)\n        for i in range(90):\n            freq = float(pbase * rng.uniform(0.98, 1.02))\n            start_s = float(obase + rng.uniform(-0.1, 0.1))\n            level = float(-20.0 + rng.uniform(-1.0, 1.0))\n            phase = float(rng.uniform(0.0, 2 * np.pi))\n            burst_s = 0.5\n            wave, floor_db = background(rng)\n            n_burst = int(burst_s * SR)\n            t = np.arange(n_burst) / SR\n            burst = (db_to_amp(level)\n                     * np.sin(2 * np.pi * freq * t + phase)).astype(np.float32)\n            edge = SR // 200\n            ramp = 0.5 - 0.5 * np.cos(\n                np.linspace(0, np.pi, edge)).astype(np.float32)\n            burst[:edge] *= ramp\n            burst[-edge:] *= ramp[::-1]\n            s0 = int(start_s * SR)\n            wave[s0:s0 + n_burst] += burst\n            split = "train" if int(order[i]) < 60 else "test"\n            finish_row(rows, f"prop_tone_{ci}_{i:02d}.wav", "tone", i, wave,\n                       PROMPTS["tone"],\n                       {"freq_hz": round(freq, 1),\n                        "start_s": round(start_s, 3), "burst_s": burst_s,\n                        "level_dbfs": round(level, 2), "gain_db": "",\n                        "speed_factor": "",\n                        "floor_dbfs": floor_db if floor_db is not None else "",\n                        "pitch_band": pband, "onset_band": oband,\n                        "split": split})\n    # Silence: the shared background scheme returns pure\n    # digital silence for most draws, so identical silence\n    # waveforms are expected and legitimate (silence is one\n    # stimulus). The discipline that matters is split\n    # hygiene: identical waveforms are grouped and a group\n    # sits entirely on one side of the split, so no stimulus\n    # is ever on both sides. Draw all 60 first, group by\n    # hash, then stamp.\n    sil_draws = []\n    for i in range(60):\n        wave, floor_db = background(rng)\n        sil_draws.append((i, wave, floor_db))\n    groups = {}\n    for i, wave, floor_db in sil_draws:\n        groups.setdefault(wave_hash(wave), []).append((i, wave, floor_db))\n    gorder = list(groups.keys())\n    rng.shuffle(gorder)\n    train_hashes = set()\n    n_train = 0\n    for h in gorder:\n        if n_train >= 40:\n            break\n        train_hashes.add(h)\n        n_train += len(groups[h])\n    for h in gorder:\n        split = "train" if h in train_hashes else "test"\n        for i, wave, floor_db in groups[h]:\n            finish_row(rows, f"prop_silence_{i:02d}.wav", "silence", i, wave,\n                       PROMPTS["silence"],\n                       {"freq_hz": "", "start_s": "", "burst_s": "",\n                        "level_dbfs": "", "gain_db": "", "speed_factor": "",\n                        "floor_dbfs": floor_db if floor_db is not None else "",\n                        "pitch_band": "", "onset_band": "",\n                        "split": split})\n    print("-" * 70)\n    tone_hashes = [r["wave_sha256"] for r in rows if r["group"] == "tone"]\n    assert len(set(tone_hashes)) == len(tone_hashes), \\\n        "duplicate tone waveform hashes; the property identities must be unique"\n    sil_by_split = {}\n    for r in rows:\n        if r["group"] == "silence":\n            sil_by_split.setdefault(r["wave_sha256"], set()).add(r["split"])\n    assert all(len(v) == 1 for v in sil_by_split.values()), \\\n        "a silence waveform sits on both sides of the split"\n    assert not (set(tone_hashes) & set(sil_by_split)), \\\n        "a tone waveform hash collides with a silence waveform"\n    print(f"generated {len(rows)} clips; {len(tone_hashes)} tone waveforms, "\n          f"all unique; silence in {len(sil_by_split)} unique waveforms, "\n          f"each entirely on one side of the split")\n    from collections import Counter\n    splits = Counter((r["group"], r["split"]) for r in rows)\n    print(f"  tone: {splits[(\'tone\', \'train\')]} train, {splits[(\'tone\', \'test\')]} test")\n    print(f"  silence: {splits[(\'silence\', \'train\')]} train, {splits[(\'silence\', \'test\')]} test")\n    def write_manifest(p):\n        with open(p, "w", newline="") as fh:\n            w = csv.DictWriter(fh, fieldnames=MANIFEST_FIELDS)\n            w.writeheader()\n            for r in rows:\n                w.writerow({k: r.get(k, "") for k in MANIFEST_FIELDS})\n    save_both("behavioral_minicpm_manifest.csv", write_manifest)\n    return rows\n\n# --- One ask ---------------------------------------------------------------------\n# One clip under one form: the waveform proof, the\n# templated feed, one manual forward, and the\n# next-token logits at the final position scored\n# exactly as Qwen\'s forward_score scored them: a full\n# softmax over the vocabulary, and each answer word\'s\n# probability is the sum over its token set. The\n# answer word is the option with the larger summed\n# probability (a tie goes to the first-named option;\n# this is the share readout, Qwen\'s correct_by_share,\n# not a generated word); the margin is the\n# first-named option\'s share of the two summed\n# probabilities, which for a yes/no form is the yes\n# share, and margin_log is log(p_a) - log(p_b).\n_PRINTED = [False]\n\ndef ask_once(path, form, expect_silence=False):\n    wave = read_wave(path, expect_silence=expect_silence)\n    sec = [rms_db_np(wave[k * SR:(k + 1) * SR]) for k in range(4)]\n    print("FEED PROOF " + json.dumps({\n        "file": os.path.basename(path),\n        "samples": int(len(wave)), "sample_rate": SR,\n        "rms_dbfs": round(rms_db_np(wave), 2),\n        "per_second_rms_dbfs": [round(v, 1) for v in sec],\n        "expect_silence": bool(expect_silence)}, sort_keys=True))\n    data = build_inputs(wave, form["question"],\n                        print_prompt=not _PRINTED[0])\n    _PRINTED[0] = True\n    out = llm_forward(data)\n    logits = out.logits[0, -1].float()\n    finite_all = bool(torch.isfinite(logits).all())\n    first_tok = processor.tokenizer.decode(\n        [int(torch.argmax(logits).item())])\n    probs = torch.softmax(logits, dim=-1)\n    pa = sum(float(probs[i].item())\n             for i in TOK[form["opt_a"]])\n    pb = sum(float(probs[i].item())\n             for i in TOK[form["opt_b"]])\n    denom = pa + pb\n    share_a = pa / denom if denom > 0 else 0.0\n    margin_log = (math.log(pa) - math.log(pb)\n                  if pa > 0 and pb > 0 else None)\n    answer = form["opt_a"] if pa >= pb else form["opt_b"]\n    del out, data\n    return {"first_token": first_tok, "pa": pa, "pb": pb,\n            "share_a": share_a, "margin_log": margin_log,\n            "answer": answer, "finite_all": finite_all}\n\nRESULT_FIELDS = ["clip", "pitch_hz", "onset_s",\n                 "pitch_band", "onset_band", "form",\n                 "question", "token_sets", "answer_word",\n                 "first_token", "truth_word", "p_yes",\n                 "p_no", "yes_share", "margin_log",\n                 "p_opt_a", "p_opt_b", "share_first",\n                 "correct_by_word", "correct_by_share"]\n\ndef run_asks(clips, label, progress_every=0):\n    # Every clip under every form, in FORMS order. A\n    # broken feed hard-stops the run (exit 7), exactly\n    # as Qwen\'s batteries stopped: a clip that was not\n    # asked is never padded and never scored.\n    rows_out = []\n    n = len(clips)\n    for i, r in enumerate(clips):\n        for form in FORMS:\n            try:\n                res = ask_once(\n                    r["file"], form,\n                    expect_silence=bool(\n                        r.get("_expect_silence", False)))\n            except FeedError as e:\n                print()\n                print("!" * 70)\n                print(f"{label} FEED HARD STOP:")\n                print(str(e))\n                print(f"Clip {r[\'clip_name\']} under form "\n                      f"{form[\'name\']} was not asked, and "\n                      f"the run stops here. No padding was "\n                      f"substituted. Show this whole "\n                      f"output to Muse.")\n                print("!" * 70)\n                sys.exit(7)\n            truth = form["truth"](r.get("pitch_band", ""),\n                                  r.get("onset_band", ""))\n            predicted = (form["opt_a"] if res["share_a"] >= 0.5\n                         else form["opt_b"])\n            row = {\n                "clip": r["clip_name"],\n                "pitch_hz": r.get("freq_hz", ""),\n                "onset_s": r.get("start_s", ""),\n                "pitch_band": r.get("pitch_band", ""),\n                "onset_band": r.get("onset_band", ""),\n                "form": form["name"],\n                "question": form["question"],\n                "token_sets": (\n                    f"{form[\'opt_a\']}={TOK[form[\'opt_a\']]}; "\n                    f"{form[\'opt_b\']}={TOK[form[\'opt_b\']]}"),\n                "answer_word": res["answer"],\n                "first_token": res["first_token"],\n                "truth_word": truth,\n                "correct_by_word": res["answer"] == truth,\n                "correct_by_share": predicted == truth,\n                "_finite": res["finite_all"],\n                "_share_a": res["share_a"]}\n            if form["kind"] == "yesno":\n                row.update({\n                    "p_yes": f"{res[\'pa\']:.6f}",\n                    "p_no": f"{res[\'pb\']:.6f}",\n                    "yes_share": f"{res[\'share_a\']:.4f}",\n                    "margin_log":\n                        (f"{res[\'margin_log\']:.4f}"\n                         if res["margin_log"] is not None\n                         else ""),\n                    "p_opt_a": "", "p_opt_b": "",\n                    "share_first": ""})\n            else:\n                row.update({\n                    "p_yes": "", "p_no": "",\n                    "yes_share": "", "margin_log": "",\n                    "p_opt_a": f"{res[\'pa\']:.6f}",\n                    "p_opt_b": f"{res[\'pb\']:.6f}",\n                    "share_first": f"{res[\'share_a\']:.4f}"})\n            rows_out.append(row)\n        if progress_every and (i + 1) % progress_every == 0:\n            print(f"  asked {i + 1}/{n} clips "\n                  f"(all {len(FORMS)} forms each)")\n        free_memory()\n    return rows_out\n\ndef auc_of(pairs):\n    # The threshold-free readout, the closers battery\'s\n    # own computation (build_closers_qwen_notebook.py,\n    # Part 2): the probability that a randomly drawn\n    # true case outranks a randomly drawn false case on\n    # the margin, ties counting one half.\n    pos = sorted(s for s, t in pairs if t)\n    neg = sorted(s for s, t in pairs if not t)\n    if not pos or not neg:\n        return 0.0\n    wins = 0.0\n    import bisect\n    for s in pos:\n        lo = bisect.bisect_left(neg, s)\n        hi = bisect.bisect_right(neg, s)\n        wins += lo + 0.5 * (hi - lo)\n    return wins / (len(pos) * len(neg))\n\ndef print_listing():\n    print("=" * 70)\n    print("OUTPUTS in " + OUT_HOME + ":")\n    for name in sorted(os.listdir(OUT_HOME)):\n        p = os.path.join(OUT_HOME, name)\n        if os.path.isfile(p):\n            print(f"  {name}  ({os.path.getsize(p)} bytes)")\n    print("Collect these through the Colab Files panel; "\n          "this notebook")\n    print("has no Drive mount and no download step, by "\n          "design.")\n\n# --- Battery flow (Path B transfer test, Nov 11, 2026) -------------------------\n# 18 real-sound presence asks, frozen in DESIGN-FROZEN-PATHB.md.\n# The question is the Series Two Test 3 form, verbatim. Each ask is\n# scored by ask_once through the manual forward (the yes share).\n# The tone cuts carried in, fixed, fitted on NO real sound:\n# default 0.5, pitch-form tone cut 0.7773, onset-form tone cut 0.4378.\nPB_DIR = "/content/pathb_clips"\nPB_CUTS = [("default", 0.5), ("pitch tone cut", 0.7773),\n           ("onset tone cut", 0.4378)]\nPB_ASKS = [\n    ("present_water_still", "swap_water.wav", "running water", "Yes"),\n    ("present_vacuum_still", "swap_vacuum.wav", "a vacuum cleaner", "Yes"),\n    ("present_beeps_still", "swap_beeps.wav", "beeping", "Yes"),\n    ("present_sizzle_still", "swap_sizzle.wav", "sizzling", "Yes"),\n    ("present_speech_still", "swap_speech.wav", "a person speaking", "Yes"),\n    ("present_mixed_vacuum", "swap_mixed.wav", "a vacuum cleaner", "Yes"),\n    ("present_water_gray", "series2_gray_water.wav", "running water", "Yes"),\n    ("present_vacuum_gray", "series2_gray_vacuum.wav", "a vacuum cleaner", "Yes"),\n    ("present_beeps_gray", "series2_gray_beeps.wav", "beeping", "Yes"),\n    ("absent_silence_water", "swap_silence.wav", "running water", "No"),\n    ("absent_silence_vacuum", "swap_silence.wav", "a vacuum cleaner", "No"),\n    ("absent_silence_beeps", "swap_silence.wav", "beeping", "No"),\n    ("absent_silence_sizzle", "swap_silence.wav", "sizzling", "No"),\n    ("absent_silence_speech", "swap_silence.wav", "a person speaking", "No"),\n    ("lure_water_asked_vacuum", "swap_water.wav", "a vacuum cleaner", "No"),\n    ("lure_vacuum_asked_beeps", "swap_vacuum.wav", "beeping", "No"),\n    ("lure_beeps_asked_water", "swap_beeps.wav", "running water", "No"),\n    ("lure_sizzle_asked_speech", "swap_sizzle.wav", "a person speaking", "No"),\n]\n\ndef pb_form(target):\n    return {"name": "presence", "kind": "yesno", "opt_a": "Yes",\n            "opt_b": "No",\n            "question": (f"Is there the sound of {target} in this "\n                         f"clip? Answer yes or no.")}\n\ndef pb_run(asks, label):\n    rows_out = []\n    for name, wav, target, truth in asks:\n        form = pb_form(target)\n        path = os.path.join(PB_DIR, wav)\n        try:\n            res = ask_once(path, form,\n                           expect_silence=(wav == "swap_silence.wav"))\n        except FeedError as e:\n            print()\n            print("!" * 70)\n            print(f"{label} FEED HARD STOP:")\n            print(str(e))\n            print(f"Ask {name} was not scored, and the run stops "\n                  f"here. No padding was substituted. Show this "\n                  f"whole output to Muse.")\n            print("!" * 70)\n            sys.exit(7)\n        predicted = "Yes" if res["share_a"] >= 0.5 else "No"\n        rows_out.append({\n            "clip": name, "pitch_hz": "", "onset_s": "",\n            "pitch_band": "", "onset_band": "", "form": "presence",\n            "question": form["question"],\n            "token_sets": f"Yes={TOK[\'Yes\']}; No={TOK[\'No\']}",\n            "answer_word": res["answer"],\n            "first_token": res["first_token"], "truth_word": truth,\n            "p_yes": f"{res[\'pa\']:.6f}", "p_no": f"{res[\'pb\']:.6f}",\n            "yes_share": f"{res[\'share_a\']:.4f}",\n            "margin_log": (f"{res[\'margin_log\']:.4f}"\n                           if res["margin_log"] is not None else ""),\n            "p_opt_a": "", "p_opt_b": "", "share_first": "",\n            "correct_by_word": res["answer"] == truth,\n            "correct_by_share": predicted == truth,\n            "_finite": res["finite_all"], "_share_a": res["share_a"]})\n        print(f"  {name}: yes_share {res[\'share_a\']:.4f}, "\n              f"answer {res[\'answer\']}, truth {truth}")\n        free_memory()\n    return rows_out\n\nif MODE == "check":\n    print("=" * 70)\n    print("SANITY: 2 Path B asks (1 present, 1 absent) through "\n          "the presence form")\n    sanity_asks = [PB_ASKS[0], PB_ASKS[9]]\n    srows = pb_run(sanity_asks, "SANITY")\n    failures = []\n    if len(srows) != len(sanity_asks):\n        failures.append(f"sanity ask count is {len(srows)}, "\n                        f"not {len(sanity_asks)}")\n    for r in srows:\n        if not r["_finite"]:\n            failures.append(f"{r[\'clip\']}: a non-finite logit in "\n                            f"the forward\'s output row")\n        m = float(r["yes_share"])\n        if not (0.0 <= m <= 1.0):\n            failures.append(f"{r[\'clip\']}: yes share {m} outside "\n                            f"[0, 1]")\n        if r["answer_word"] not in ("Yes", "No"):\n            failures.append(f"{r[\'clip\']}: answer word "\n                            f"{r[\'answer_word\']!r} outside Yes/No")\n    if failures:\n        print()\n        print("!" * 70)\n        print("SANITY: FAIL")\n        for f in failures:\n            print("  -", f)\n        print("The battery must not run on a feed whose sanity "\n              "asks fail.")\n        print("Show this whole output to Muse.")\n        print("!" * 70)\n        sys.exit(12)\n    print("SANITY: PASS - 2 scored asks; every logit finite, "\n          "every yes share")\n    print("inside [0, 1], every answer word inside Yes/No.")\n    print_listing()\n    print("check mode done: proof, load, token resolution, and "\n          "the sanity asks")\n    print("all passed. Run Step 4 for the battery.")\n    sys.exit(0)\n\nprint("=" * 70)\nprint("BATTERY: 18 Path B transfer asks (9 present, 9 absent), "\n      "presence form,")\nprint("scored by the yes share from the manual forward.")\nresult_rows = pb_run(PB_ASKS, "BATTERY")\nassert len(result_rows) == 18, \\\n    f"expected 18 result rows, got {len(result_rows)}"\n\ndef write_results(p):\n    import csv\n    with open(p, "w", newline="") as fh:\n        w = csv.DictWriter(fh, fieldnames=RESULT_FIELDS,\n                           extrasaction="ignore")\n        w.writeheader()\n        w.writerows(result_rows)\nsave_both("pathb_minicpm_results.csv", write_results)\n\nprint("BATTERY READOUT (mechanical):")\npairs = [(float(r["yes_share"]), r["truth_word"] == "Yes")\n         for r in result_rows]\nprint(f"  AUC of the yes share, present vs absent: "\n      f"{auc_of(pairs):.4f} ({len(result_rows)} asks)")\nfor cut_name, cut in PB_CUTS:\n    right = sum(1 for r in result_rows\n                if ("Yes" if float(r["yes_share"]) >= cut else "No")\n                == r["truth_word"])\n    print(f"  accuracy at the {cut_name} cut ({cut}): "\n          f"{right}/{len(result_rows)} = {right / len(result_rows):.4f}")\nprint("=" * 70)\nprint("battery done: 18 scored asks, yes shares saved in "\n      "pathb_minicpm_results.csv.")\nprint("Outputs: pathb_minicpm_results.csv, "\n      "pathb_minicpm_run_log.txt.")\nprint_listing()\n'
with open("/content/minicpm_behavioral_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/minicpm_behavioral_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, load, token resolution, sanity asks ---------------------------
import subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_behavioral_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, the load, the token resolution, clip")
    print("generation, a feed proof, or the sanity asks. Do not run Step 4.")
    print("Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, answer tokens")
print("resolved, clips generated and measured, and the sanity asks")
print("passed (SANITY: PASS in the runner's output). Go on to Step 4.")

## Step 4 · Run the battery: 360 tones, six forms, 2,160 scored asks

The next cell runs the runner in **battery mode**:
proof, load, token resolution, and clip generation
again (a fresh load re-proves everything), then the
asks.

**The asks.** Each of the 360 factorial tones is fed
once per form, six manual forwards per clip, 2,160 in
all. Every feed prints its proof before its forward.
For each ask the runner records: the clip, its true
pitch in Hz and true onset in seconds with their
bands, the form's name and its exact question text,
the answer word (the option whose summed probability
is larger), the token id sets used, the first token
of the full next-token row, the true answer word, and
the scoring probabilities. Yes/no forms record p_yes
and p_no (each the sum of the full-softmax
probabilities over the word's token set), the yes
share, and margin_log (log p_yes minus log p_no);
choice forms record both options' probabilities and
the first-named option's share. Two correctness
columns ride along:
correct by word, and correct by margin at the fixed
0.5 cut. Results land in
`behavioral_minicpm_results.csv`; the stimulus record
is `behavioral_minicpm_manifest.csv`; everything the
runner printed is in `pathb_minicpm_run_log.txt`.

**The tail.** The runner ends with a mechanical
readout per form: accuracy by word, accuracy by margin
at 0.5, the AUC of the margin, the mean margin on each
truth class, and the answer-word counts. For the
yes/no forms it also prints the AUC of margin_log,
which ranks identically to the share's by
construction. The readout makes no claims; the
predictions (P1, P2, P3, at the top of this notebook)
are scored against it afterwards, from the CSV.

In [ ]:
# --- Battery mode: the MiniCPM behavioral battery ---------------------------------------
import subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_behavioral_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: the token resolution failed, a feed proof")
    print("hard-stopped the run, or clip generation failed. Show it")
    print("to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: 2,160 scored asks recorded, results and manifest")
print("saved in /content/project_k_outputs.")

## Closing · How to read the answers

Read the readout per form, and keep the three scores
separate, because they answer different questions:

- **By word** is what the model would have said. On
  Qwen, the spoken yes/no answers defaulted to the
  affirmative or first-named option in every wording
  tried, which is why P1 and P3 are registered the way
  they are.
- **By margin at 0.5** is the same lean read at the
  fixed cut the field would naively use.
- **AUC** is the threshold-free question: does the
  lean itself separate the truth, whatever cut is
  chosen? On Qwen the original wordings' margins
  separated pitch perfectly (AUC 1.0000) and onset
  nearly perfectly (0.9988) while the spoken answers
  sat at their base rates: the loss was the cut's
  placement, not the representation. P2 asks whether
  MiniCPM's margin does the same. If it does, the two
  models' reporting losses share a shape; if MiniCPM's
  AUC is low where its words fail, its loss sits
  earlier than Qwen's, and the comparison page says so
  either way.
- **Scope.** Every claim stays capped at this model,
  these tones, these six wordings, this run. A margin
  is a property of a prompt as much as of a model:
  claims name their wording, and the reversed-pitch
  lesson from Qwen's closers (one wording's margin can
  degrade to 0.6331 while another's is perfect) is the
  standing warning against pooling margins across
  forms.

When you finish, collect the three files from
`/content/project_k_outputs` through the Colab Files
panel: `behavioral_minicpm_results.csv` (the 2,160
asks), `behavioral_minicpm_manifest.csv` (the clips,
their true pitch and onset, and their waveform
hashes), and `pathb_minicpm_run_log.txt` (the
whole run, including the token resolution table and
the first feed's full prompt text). Send them back,
and the cell 3 page of the MiniCPM comparison against
Qwen's frozen ledger gets written.